<a href="https://colab.research.google.com/github/darshita27-cmd/Music-Recommender/blob/main/RS_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy scikit-learn surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2611309 sha256=b7dd308a0b4fa0d2c69491c43313651d23eaf567f4cf5000ed941a1bb37eaf84
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [17]:
pip install streamlit scikit-surprise pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.4 MB/s eta 0:00:00


In [19]:
import streamlit as st
import pandas as pd # for data manipulation, cleaning, analysis
from surprise import SVD, Dataset, Reader # surprise (simple python recommendation system engine) mainly focusing on collaborative filtering. SVD foro matrix factorization and decompose the user item rating into lower dimentional factor. Dataset for loading and structuring data in surprise's format. Reader--how to read rating data (eg: scale of 1 to 5 starts, or ignore certain columns)
from surprise.model_selection import train_test_split
from surprise import accuracy
import os

In [18]:
@st.cache_data # streamlit decorator that chaches function's return value based on its input and if same argumemt is called again than than it will return the cached result instead of re-executing the function
def load_data():
  df=pd.read_csv('/content/Last.fm_data.csv')
  df=df.head(50000)
  print(df.head())
  play_counts = df.groupby(['Username', 'Track']).size().reset_index(name='Playcount') # playcount = username X track (implicit rating)
  return play_counts
play_counts=load_data()
@st.cache_resourse
def train_model(data):
  reader=Reader(rating_scale=(0,play_counts['Playcount'].max())) # Reader is a class that defines format of rating data. rating scale is 0 to playcount ( playcount is no. of items an item was interacted with)
  data=Dataset.load_from_df(play_counts[['Username','Track','Playcount']],reader)
  trainset,testset=train_test_split(data,test_size=0.2)
  model=SVD()
  model.fit(trainset)
  return model
model=train_model(play_counts)
# predictions=model.test(testset)
# rmse=accuracy.rmse(predictions)
# print(f"model RMSE:{rmse}")   # root mean square error
def recommend_songs(user_id,n=5):
  all_songs=play_counts['Track'].unique() # from track_name column finding out unique values
  listened=play_counts[play_counts['Username']==user_id]['Track'].values #finding songs that are already listened by the user
  predictions=[(song,model.predict(user_id,song).est) for song in all_songs if song not in listened] # predictions will be a list of tuple in format (song_name,estimated_rating. model is SVD (as above). predict(user_id,song) generates a prediction for how much used_id would like the song.  .est extracts estimated rating
  predictions.sort(key=lambda x:x[1],reverse=True) # lambda function will be taking tuple (eg (song A,8.5)) and will return its second element x[1] which will help sorting based on ratings and not song names.reverse=True is for descending order
  return predictions[:n]
  # print(f"\n Top {n} Recommended songs for user {user_id}:")
  # for song, score in predictions[:n]:
    #print(f"{song} (predicted Score: {score:.2f})")
#recommend_songs(user_id=play_counts['Username'].iloc[0]) # iloc[0] to get first value from column. iloc is basically panda indexing

# steamlit UI

st.title("🎧 Music recommender system (collaborative Filtering)")
st.markdown("Get personalized song recommendations using SVD model")
# sidebar
usernames=sorted(play_counts['Username'].unique())
selected_user=st.sidebar.selectbox("Select a user:",usernames)
num_recs=st.sidebar.slider("Number of recommendations",3,15,5)
# recommend button
if st.button("Recommend Songs"):
  with st.spinner("Generating recommendations..."):
    recs=recommend_songs(selected_user, n=num_recs)
  st.subheader(f"🎶 Top{num_recs} Recommendations for {selected_user}")
  for i , (track,score) in enumerate(recs,1):
    st.write(f"**{i}.{track}**-Predicted Rating: {score:.2f}")
st.markdown("---")

NameError: name 'st' is not defined